# PTM-LLaMA Evaluation Notebook

This notebook evaluates the multi-PTM `ptm-llama` model on three held-out tasks: methylation, phosphorylation, and ubiquitination prediction. The model is loaded from the Hugging Face Hub; calibration and test partitions are re-derived from the source dataset using the same protein-level split (seed and ratios) as the training notebook.

## Methodology

For each PTM type, full-protein inference proceeds in three steps:

1. **Sliding windows.** A 21-residue window is slid across the protein with stride 5; a tail window is appended so the final residues are covered.
2. **Per-window generation.** The model is prompted with the PTM-type-specific instruction. Each predicted residue letter is validated against the window sequence at the indicated position; mismatches are discarded. Validated predictions are mapped to full-protein coordinates and accumulated into a per-residue **consensus score**, defined as `(# windows predicting the residue as a site) / (# windows covering the residue)`.
3. **Thresholding.** A PTM-type-specific F1-optimal threshold is derived from the held-out calibration set and applied at test time.

## Calibration and test sets

Deriving an operating threshold from the same data on which the headline metrics are reported produces optimistically biased estimates. Two disjoint protein-level held-out sets are used:

1. **Calibration set** — 10% of the source proteins, held out from training. For each PTM type, the F1-optimal threshold over the per-residue consensus ROC is selected from this set.
2. **Test set** — a disjoint 10% of the source proteins. The calibration-derived per-PTM-type thresholds are applied without modification; the metrics reported are unbiased point estimates of generalization.

The per-PTM-type thresholds and windowing parameters are persisted to the model repository as `inference_config.json` for use by downstream inference code.

A separate **cross-instruction ablation** assesses whether the instruction prompt has the intended effect: each test sequence is prompted with all three PTM-type instructions, and the resulting outputs are compared against each PTM type's ground-truth labels. A well-trained instruction-tuned model should produce predictions whose accuracy for PTM type T is highest when the prompt requests PTM type T.

## 1. Environment Setup

In [ ]:
!pip install -q -U \
    "transformers>=4.44,<4.50" \
    "peft>=0.11" \
    "accelerate>=0.30" \
    "huggingface_hub>=0.24" \
    scikit-learn matplotlib seaborn "pandas<3" sentencepiece

# peft >=0.11 errors out if torchao is installed at an incompatible version.
# Colab ships torchao==0.10.0 pre-installed; we don't use it, so remove it.
!pip uninstall -y -q torchao

In [ ]:
import os, re, json, math, gc, time
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi, login
from sklearn.metrics import (
    roc_curve, roc_auc_score, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
)

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configuration

The PTM-type instruction templates and the protein-level split parameters must match the values used by the training notebook. They are re-stated here so the evaluation notebook is self-contained, but they should never drift from the training notebook's values; the source-of-truth is the training notebook.

In [ ]:
HF_REPO_ID = 'jbenbudd/ptm-llama'
PTM_DATA_CSV = 'datasets/all_ptm_sites_site_level.csv'

# Must match the training notebook exactly.
SPLIT_SEED = 42
TRAIN_RATIO = 0.80
CAL_RATIO = 0.10
TEST_RATIO = 0.10

WINDOW_SIZE = 21
STRIDE = 5
MAX_NEW_TOKENS = 64
BATCH_SIZE = 128

PTM_INSTRUCTIONS = {
    'Methylation':     '[Predict the methylation sites given the peptide sequence]',
    'Phosphorylation': '[Predict the phosphorylation sites given the peptide sequence]',
    'Ubiquitination':  '[Predict the ubiquitination sites given the peptide sequence]',
}

# Canonical target residues per PTM type, used in the per-residue-type breakdown.
PTM_CANONICAL_RESIDUES = {
    'Methylation':     ['K', 'R'],
    'Phosphorylation': ['S', 'T', 'Y'],
    'Ubiquitination':  ['K'],
}

# Cross-instruction ablation runs the full sliding-window inference for every
# (test protein, prompted PTM type) pair. To keep the ablation tractable we
# sub-sample test proteins; set ABLATION_PROTEIN_LIMIT to None to run on all.
ABLATION_PROTEIN_LIMIT = 200

EVAL_DIR = 'eval_outputs'
os.makedirs(EVAL_DIR, exist_ok=True)

PUSH_MODEL_CARD = True

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_API_TOKEN')
except ImportError:
    HF_TOKEN = os.environ.get('HF_API_TOKEN')

assert HF_TOKEN, 'No HF token found. Set HF_API_TOKEN in Colab Secrets or env vars.'
login(token=HF_TOKEN)
print('Logged in to Hugging Face.')

## 3. Load Model + Tokenizer from Hugging Face

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='sdpa',
)
model.eval()
model.config.use_cache = True

if torch.cuda.is_available():
    dev = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(dev)
    total_gb = props.total_memory / 1024**3
    alloc_gb = torch.cuda.memory_allocated(dev) / 1024**3
    print(f'Loaded {HF_REPO_ID}')
    print(f'  GPU              : {props.name}')
    print(f'  Compute cap.     : sm_{props.major}{props.minor}')
    print(f'  VRAM             : {alloc_gb:.2f} GiB used / {total_gb:.1f} GiB total')
    print(f'  Model dtype      : {next(model.parameters()).dtype}')
    print(f'  Attn impl        : {getattr(model.config, "_attn_implementation", "default")}')
    print(f'  use_cache        : {model.config.use_cache}')
    print(f'  Eval batch size  : {BATCH_SIZE}')
else:
    print(f'Loaded {HF_REPO_ID} (CPU)')

## 4. Sliding-Window Inference Helpers

Five helper functions used by the calibration, test, and ablation sections:

- `make_windows(seq)` — produces sliding-window starts (stride 5) with a tail window so the final residues are covered.
- `build_prompt(window_seq, ptm_type)` — formats a window into the Alpaca prompt used during training, with the instruction selected by PTM type.
- `generate_batch(prompts)` — batched greedy generation; returns only the newly generated completions.
- `extract_sites(text)` — parses `Sites=<R5,D12,...>` from a completion and returns `[(letter, window_position), ...]`.
- `run_inference_on_protein(seq, ptm_type)` — runs sliding-window inference on a full protein for one PTM type and returns per-residue consensus arrays (`covered`, `predicted`, `y_score`).

In [ ]:
PROMPT_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n{input}\n\n'
    '### Response:\n'
)

SITE_RE  = re.compile(r'^([A-Z])(\d+)$')
SITES_RE = re.compile(r'Sites=<([^>]*)>')


def make_windows(seq: str, w: int = WINDOW_SIZE, s: int = STRIDE) -> List[Tuple[int, str]]:
    L = len(seq)
    if L <= w:
        return [(0, seq)]
    starts = list(range(0, L - w + 1, s))
    if starts[-1] + w < L:
        starts.append(L - w)
    return [(i, seq[i:i + w]) for i in starts]


def build_prompt(window_seq: str, ptm_type: str) -> str:
    return PROMPT_TEMPLATE.format(
        instruction=PTM_INSTRUCTIONS[ptm_type],
        input=f'Seq=<{window_seq}>',
    )


def extract_sites(text: str) -> List[Tuple[str, int]]:
    m = SITES_RE.search(text)
    if not m:
        return []
    body = m.group(1).strip()
    if body == '':
        return []
    out = []
    for part in body.split(','):
        mm = SITE_RE.match(part.strip())
        if mm:
            out.append((mm.group(1), int(mm.group(2))))
    return out


@torch.inference_mode()
def generate_batch(prompts: List[str]) -> List[str]:
    enc = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    new_tokens = out[:, enc.input_ids.shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)


def run_inference_on_proteins(
    proteins: List[Dict],
    ptm_type: str,
    desc: str = 'inference',
):
    """Run sliding-window inference for one PTM type across many proteins.

    Each protein dict must have `sequence` (string). The function adds three
    arrays to each protein dict: `covered`, `predicted`, and `y_score`.
    """
    jobs = []
    for prot_idx, p in enumerate(proteins):
        L = len(p['sequence'])
        p['covered']   = np.zeros(L, dtype=np.int32)
        p['predicted'] = np.zeros(L, dtype=np.int32)
        for start, w_seq in make_windows(p['sequence']):
            jobs.append((prot_idx, start, w_seq))

    t0 = time.time()
    for i in tqdm(range(0, len(jobs), BATCH_SIZE), desc=desc):
        batch = jobs[i:i + BATCH_SIZE]
        prompts = [build_prompt(w_seq, ptm_type) for _, _, w_seq in batch]
        outs = generate_batch(prompts)
        for (prot_idx, start, w_seq), comp in zip(batch, outs):
            p = proteins[prot_idx]
            seq, L = p['sequence'], len(p['sequence'])
            end = min(start + len(w_seq), L)
            p['covered'][start:end] += 1
            for letter, pos_local in extract_sites(comp):
                if not (1 <= pos_local <= len(w_seq)):
                    continue
                pos_full_0 = start + pos_local - 1
                if not (0 <= pos_full_0 < L):
                    continue
                if seq[pos_full_0] != letter:
                    continue
                p['predicted'][pos_full_0] += 1

    for p in proteins:
        mask = p['covered'] > 0
        p['mask'] = mask
        p['y_score'] = np.zeros(len(p['sequence']), dtype=np.float32)
        p['y_score'][mask] = p['predicted'][mask] / p['covered'][mask]

    print(f'  {desc}: {len(jobs):,} windows across {len(proteins):,} proteins in {time.time() - t0:.1f}s')


# Quick sanity check.
_t = 'CQIVLTPELEGVEFALPKITR'
print('Sample generations:')
for ptm in PTM_INSTRUCTIONS:
    out = generate_batch([build_prompt(_t, ptm)])[0].strip()
    print(f'  [{ptm:<16}] {_t}  ->  {out}')

## 5. Data Loading and Split Reconstruction

The same protein-level split is re-derived from `all_ptm_sites_site_level.csv` using the same seed and ratios as the training notebook. The training partition is then discarded — the evaluation notebook only operates on the calibration and test partitions.

In [ ]:
df = pd.read_csv(PTM_DATA_CSV)
print(f'Loaded {len(df):,} annotated sites from {PTM_DATA_CSV}')
print()
print('Sites per PTM type:')
print(df['PTM_Type'].value_counts().to_string())

In [ ]:
# Re-derive the same protein-level split as the training notebook.
unique_proteins = np.array(sorted(df['uniprot_id'].unique()))
rng = np.random.default_rng(SPLIT_SEED)
shuffled = rng.permutation(unique_proteins)

n_total = len(shuffled)
n_train = int(n_total * TRAIN_RATIO)
n_cal = int(n_total * CAL_RATIO)

train_proteins = set(shuffled[:n_train])
cal_proteins   = set(shuffled[n_train:n_train + n_cal])
test_proteins  = set(shuffled[n_train + n_cal:])

print(f'Re-derived split (seed = {SPLIT_SEED}):')
print(f'  train:       {len(train_proteins):,}')
print(f'  calibration: {len(cal_proteins):,}')
print(f'  test:        {len(test_proteins):,}')

In [ ]:
# Build per-(protein, PTM_type) records for calibration and test partitions.
groups = (
    df.drop_duplicates(subset=['uniprot_id', 'PTM_Type'])
      [['uniprot_id', 'PTM_Type', 'protein_sequence', 'binary_mask']]
      .reset_index(drop=True)
)


def to_record(row):
    seq = str(row.protein_sequence)
    mask = str(row.binary_mask)
    if len(seq) != len(mask):
        return None
    y_true = np.zeros(len(seq), dtype=np.int8)
    for i, c in enumerate(mask):
        if c == '1':
            y_true[i] = 1
    return {
        'uniprot_id': row.uniprot_id,
        'ptm_type': row.PTM_Type,
        'sequence': seq,
        'y_true': y_true,
    }


cal_records: Dict[str, List[Dict]] = {ptm: [] for ptm in PTM_INSTRUCTIONS}
test_records: Dict[str, List[Dict]] = {ptm: [] for ptm in PTM_INSTRUCTIONS}

for row in groups.itertuples(index=False):
    rec = to_record(row)
    if rec is None:
        continue
    if row.uniprot_id in cal_proteins:
        cal_records[row.PTM_Type].append(rec)
    elif row.uniprot_id in test_proteins:
        test_records[row.PTM_Type].append(rec)

print('Calibration (protein, PTM-type) records:')
for ptm, recs in cal_records.items():
    print(f'  {ptm:<16} {len(recs):>6,}')
print('Test (protein, PTM-type) records:')
for ptm, recs in test_records.items():
    print(f'  {ptm:<16} {len(recs):>6,}')

## 6. Calibration: Per-PTM-Type Threshold Derivation

For each PTM type, calibration is performed independently. Inference is run with the PTM-type's instruction on every sliding window of every calibration protein annotated with that PTM type. The per-residue consensus scores are pooled across proteins and the F1-maximizing threshold over the resulting ROC is selected as the operating threshold for that PTM type.

In [ ]:
calibration_results: Dict[str, dict] = {}

for ptm_type, recs in cal_records.items():
    if not recs:
        print(f'No calibration data for {ptm_type}, skipping.')
        continue
    print(f'\n=== Calibrating {ptm_type} ({len(recs):,} proteins) ===')
    run_inference_on_proteins(recs, ptm_type, desc=f'cal/{ptm_type}')

    y_true = np.concatenate([r['y_true'][r['mask']]  for r in recs]).astype(np.int8)
    y_score = np.concatenate([r['y_score'][r['mask']] for r in recs]).astype(np.float32)
    residues = np.concatenate([
        np.frombuffer(r['sequence'].encode('ascii'), dtype=np.uint8)[r['mask']] for r in recs
    ])

    if y_true.sum() == 0:
        print(f'No positive examples for {ptm_type} in calibration; skipping threshold derivation.')
        continue

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    candidates = np.unique(y_score)
    if len(candidates) > 500:
        candidates = np.linspace(0, 1, 501)

    sweep = []
    for t in candidates:
        yp = (y_score >= t).astype(np.int8)
        p, r, f, _ = precision_recall_fscore_support(y_true, yp, average='binary', zero_division=0)
        sweep.append((float(t), f, p, r))
    best_idx = int(np.argmax([s[1] for s in sweep]))
    best_t, best_f1, best_p, best_r = sweep[best_idx]

    calibration_results[ptm_type] = {
        'records': recs,
        'y_true': y_true,
        'y_score': y_score,
        'residues': residues,
        'fpr': fpr,
        'tpr': tpr,
        'auc': float(auc),
        'sweep': sweep,
        'threshold': float(best_t),
        'f1': float(best_f1),
        'precision': float(best_p),
        'recall': float(best_r),
        'n_residues': int(len(y_true)),
        'prevalence': float(y_true.mean()),
    }

    print(f'  residues evaluated   : {len(y_true):,}')
    print(f'  positive prevalence  : {y_true.mean() * 100:.3f}%')
    print(f'  ROC AUC              : {auc:.4f}')
    print(f'  F1-optimal threshold : {best_t:.4f}')
    print(f'    precision @ t      : {best_p:.4f}')
    print(f'    recall    @ t      : {best_r:.4f}')
    print(f'    F1        @ t      : {best_f1:.4f}')

LOCKED_THRESHOLDS = {ptm: r['threshold'] for ptm, r in calibration_results.items()}
print('\nLocked thresholds:', LOCKED_THRESHOLDS)

In [ ]:
# Calibration ROC + threshold sweep figure (one column per PTM type).
ptm_types_with_results = list(calibration_results.keys())
n = len(ptm_types_with_results)
fig, axes = plt.subplots(2, n, figsize=(5.5 * n, 9))
if n == 1:
    axes = np.array([[axes[0]], [axes[1]]])

for col, ptm in enumerate(ptm_types_with_results):
    r = calibration_results[ptm]
    ax_roc = axes[0, col]
    ax_roc.plot(r['fpr'], r['tpr'], linewidth=2, label=f'ROC (AUC={r["auc"]:.3f})')
    ax_roc.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
    ax_roc.set_xlabel('False positive rate')
    ax_roc.set_ylabel('True positive rate')
    ax_roc.set_title(f'{ptm}: calibration ROC')
    ax_roc.legend(loc='lower right')
    ax_roc.grid(True, alpha=0.3)

    ts, f1s, ps, rs = zip(*r['sweep'])
    ax_sw = axes[1, col]
    ax_sw.plot(ts, f1s, label='F1', linewidth=2)
    ax_sw.plot(ts, ps, label='Precision', alpha=0.7)
    ax_sw.plot(ts, rs, label='Recall', alpha=0.7)
    ax_sw.axvline(r['threshold'], linestyle='--', color='red', alpha=0.7,
                  label=f'Locked t = {r["threshold"]:.3f}')
    ax_sw.set_xlabel('Consensus threshold')
    ax_sw.set_ylabel('Metric')
    ax_sw.set_title(f'{ptm}: threshold sweep')
    ax_sw.legend()
    ax_sw.grid(True, alpha=0.3)

fig.tight_layout()
cal_plot_path = os.path.join(EVAL_DIR, 'calibration_threshold_sweep.png')
fig.savefig(cal_plot_path, dpi=150)
plt.show()
print('Saved:', cal_plot_path)

## 7. Test Set Evaluation

For each PTM type, sliding-window inference is run on every test protein annotated with that PTM type, the per-residue consensus scores are computed, and the calibration-derived threshold is applied to produce the final binary site calls. No threshold selection is performed on the test set; the metrics reported are unbiased point estimates of generalization.

In [ ]:
test_results: Dict[str, dict] = {}

for ptm_type, recs in test_records.items():
    if not recs:
        print(f'No test data for {ptm_type}, skipping.')
        continue
    if ptm_type not in LOCKED_THRESHOLDS:
        print(f'No calibration threshold for {ptm_type}, skipping test eval.')
        continue
    print(f'\n=== Testing {ptm_type} ({len(recs):,} proteins) ===')
    run_inference_on_proteins(recs, ptm_type, desc=f'test/{ptm_type}')

    y_true = np.concatenate([r['y_true'][r['mask']]  for r in recs]).astype(np.int8)
    y_score = np.concatenate([r['y_score'][r['mask']] for r in recs]).astype(np.float32)
    residues = np.concatenate([
        np.frombuffer(r['sequence'].encode('ascii'), dtype=np.uint8)[r['mask']] for r in recs
    ])

    t = LOCKED_THRESHOLDS[ptm_type]
    y_pred = (y_score >= t).astype(np.int8)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    accuracy    = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision   = tp / max(tp + fp, 1) if (tp + fp) else 0.0
    recall      = tp / max(tp + fn, 1) if (tp + fn) else 0.0
    specificity = tn / max(tn + fp, 1) if (tn + fp) else 0.0
    f1          = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    try:
        auc = roc_auc_score(y_true, y_score) if y_true.sum() > 0 else float('nan')
    except ValueError:
        auc = float('nan')

    test_results[ptm_type] = {
        'records': recs,
        'y_true': y_true,
        'y_score': y_score,
        'y_pred': y_pred,
        'residues': residues,
        'threshold': float(t),
        'auc': float(auc) if not math.isnan(auc) else None,
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'specificity': float(specificity),
        'f1': float(f1),
        'confusion_matrix': {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)},
        'n_residues': int(len(y_true)),
        'prevalence': float(y_true.mean()),
    }

    print(f'  residues evaluated  : {len(y_true):,}')
    print(f'  positive prevalence : {y_true.mean() * 100:.3f}%')
    print(f'  threshold (locked)  : {t:.4f}')
    print(f'  Accuracy            : {accuracy:.4f}')
    print(f'  Precision           : {precision:.4f}')
    print(f'  Recall (sensitivity): {recall:.4f}')
    print(f'  Specificity         : {specificity:.4f}')
    print(f'  F1                  : {f1:.4f}')
    if not math.isnan(auc):
        print(f'  ROC AUC             : {auc:.4f}  (threshold-free)')
    print(f'  Confusion: TN={tn:,} FP={fp:,} FN={fn:,} TP={tp:,}')

In [ ]:
# Per-PTM-type test confusion matrices and ROC curves.
ptm_types_with_test = list(test_results.keys())
n = len(ptm_types_with_test)
fig, axes = plt.subplots(2, n, figsize=(5.5 * n, 9))
if n == 1:
    axes = np.array([[axes[0]], [axes[1]]])

for col, ptm in enumerate(ptm_types_with_test):
    r = test_results[ptm]
    cm = np.array([
        [r['confusion_matrix']['TN'], r['confusion_matrix']['FP']],
        [r['confusion_matrix']['FN'], r['confusion_matrix']['TP']],
    ])
    sns.heatmap(
        cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=axes[0, col],
        xticklabels=['Pred: non-site', 'Pred: site'],
        yticklabels=['True: non-site', 'True: site'],
    )
    axes[0, col].set_title(f'{ptm}: confusion @ t = {r["threshold"]:.3f}')

    if r['auc'] is not None:
        fpr, tpr, _ = roc_curve(r['y_true'], r['y_score'])
        axes[1, col].plot(fpr, tpr, linewidth=2, label=f'Test ROC (AUC = {r["auc"]:.3f})')
    axes[1, col].plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='Random')
    op_fpr = 1.0 - r['specificity']
    op_tpr = r['recall']
    axes[1, col].scatter([op_fpr], [op_tpr], color='red', s=80, zorder=3,
                         label=f'Locked operating point\n(t = {r["threshold"]:.3f})')
    axes[1, col].set_xlabel('False positive rate')
    axes[1, col].set_ylabel('True positive rate')
    axes[1, col].set_title(f'{ptm}: test ROC')
    axes[1, col].legend(loc='lower right')
    axes[1, col].grid(True, alpha=0.3)

fig.tight_layout()
test_plot_path = os.path.join(EVAL_DIR, 'test_metrics.png')
fig.savefig(test_plot_path, dpi=150)
plt.show()
print('Saved:', test_plot_path)

## 8. Per-Residue-Type Breakdown

For each PTM type, the metrics are broken down by the canonical target residues (e.g., S/T/Y for phosphorylation, K for ubiquitination, K/R for methylation). The breakdown is evaluated at the calibration-derived threshold for that PTM type.

In [ ]:
per_residue_rows = []
for ptm_type, r in test_results.items():
    for aa in PTM_CANONICAL_RESIDUES.get(ptm_type, []):
        mask = r['residues'] == ord(aa)
        if mask.sum() == 0:
            continue
        yt = r['y_true'][mask]
        ys = r['y_score'][mask]
        yp = (ys >= r['threshold']).astype(np.int8)
        cm_aa = confusion_matrix(yt, yp, labels=[0, 1])
        tn_a, fp_a, fn_a, tp_a = cm_aa.ravel()
        acc_a = (tp_a + tn_a) / max(tp_a + tn_a + fp_a + fn_a, 1)
        p_a = tp_a / max(tp_a + fp_a, 1) if (tp_a + fp_a) else 0.0
        r_a = tp_a / max(tp_a + fn_a, 1) if (tp_a + fn_a) else 0.0
        s_a = tn_a / max(tn_a + fp_a, 1) if (tn_a + fp_a) else 0.0
        f_a = 2 * p_a * r_a / (p_a + r_a) if (p_a + r_a) else 0.0
        try:
            auc_a = roc_auc_score(yt, ys) if yt.sum() > 0 else float('nan')
        except ValueError:
            auc_a = float('nan')
        per_residue_rows.append({
            'ptm_type': ptm_type,
            'residue': aa,
            'n_residues': int(mask.sum()),
            'n_positive': int(yt.sum()),
            'auc': auc_a,
            'accuracy': acc_a,
            'precision': p_a,
            'recall': r_a,
            'specificity': s_a,
            'f1': f_a,
        })
per_residue_df = pd.DataFrame(per_residue_rows)
per_residue_df

## 9. Cross-Instruction Ablation

This section measures whether the instruction prompt has the intended effect. For a sub-sample of test proteins, sliding-window inference is run three times — once per PTM-type instruction — on the same protein sequence. The resulting per-residue consensus scores are then evaluated against each PTM type's ground truth.

Two quantities matter:

1. **Within-task vs. cross-task AUC.** For each pair `(true PTM type T, prompted PTM type P)`, AUC is computed against T's ground-truth labels. A working instruction-tuned model should have higher AUC on the diagonal (P = T) than off-diagonal (P ≠ T).
2. **Instruction-following rate.** The fraction of windows whose generated `Sites=<...>` output differs across the three PTM-type prompts. A rate near 0 would indicate the model ignores the instruction; a rate of 1 indicates strict instruction-conditioning.

In [ ]:
# Sub-sample test proteins for the ablation. Only consider proteins that have
# annotations of at least one PTM type (to compute meaningful within-task AUC).
all_test_protein_ids = list(test_proteins)
rng_ablation = np.random.default_rng(SPLIT_SEED + 1)
shuffled_test = rng_ablation.permutation(all_test_protein_ids)
if ABLATION_PROTEIN_LIMIT is not None:
    selected_test_ids = set(shuffled_test[:ABLATION_PROTEIN_LIMIT].tolist())
else:
    selected_test_ids = set(shuffled_test.tolist())

# Build a per-protein record carrying ground-truth masks for all PTM types
# the protein is annotated with.
ablation_proteins: Dict[str, Dict] = {}
for row in groups.itertuples(index=False):
    if row.uniprot_id not in selected_test_ids:
        continue
    if row.uniprot_id not in ablation_proteins:
        ablation_proteins[row.uniprot_id] = {
            'uniprot_id': row.uniprot_id,
            'sequence': str(row.protein_sequence),
            'masks': {},
        }
    seq = str(row.protein_sequence)
    mask_str = str(row.binary_mask)
    if len(seq) != len(mask_str):
        continue
    y_true = np.zeros(len(seq), dtype=np.int8)
    for i, c in enumerate(mask_str):
        if c == '1':
            y_true[i] = 1
    ablation_proteins[row.uniprot_id]['masks'][row.PTM_Type] = y_true

ablation_list = list(ablation_proteins.values())
print(f'Cross-instruction ablation will run on {len(ablation_list)} proteins.')
print(f'Total prompted runs: {len(ablation_list)} proteins x {len(PTM_INSTRUCTIONS)} PTM types = '
      f'{len(ablation_list) * len(PTM_INSTRUCTIONS)} protein-PTM passes.')

In [ ]:
# Run inference for each prompted PTM type on the same set of proteins.
# Each pass populates p['y_score'] (and p['covered']/p['predicted']); store
# them indexed by prompted PTM type for the cross-task comparison.
ablation_predictions: Dict[str, List[np.ndarray]] = {ptm: [] for ptm in PTM_INSTRUCTIONS}
ablation_masks: Dict[str, List[np.ndarray]] = {ptm: [] for ptm in PTM_INSTRUCTIONS}
ablation_window_outputs: Dict[Tuple[str, int, int], Dict[str, str]] = {}

for prompted_ptm in PTM_INSTRUCTIONS:
    print(f'\n=== Ablation: prompted PTM = {prompted_ptm} ===')
    # Build a fresh per-protein structure for this prompt pass.
    pass_records = [{'sequence': p['sequence']} for p in ablation_list]
    run_inference_on_proteins(pass_records, prompted_ptm, desc=f'ablation/{prompted_ptm}')
    for p, pr in zip(ablation_list, pass_records):
        ablation_predictions[prompted_ptm].append(pr['y_score'][pr['mask']])
        ablation_masks[prompted_ptm].append(pr['mask'])

# Cross-task AUC matrix: rows = true PTM type (where ground truth exists),
# cols = prompted PTM type.
cross_auc = pd.DataFrame(
    index=list(PTM_INSTRUCTIONS),
    columns=list(PTM_INSTRUCTIONS),
    dtype=float,
)
for true_ptm in PTM_INSTRUCTIONS:
    for prompted_ptm in PTM_INSTRUCTIONS:
        yts, yss = [], []
        for i, p in enumerate(ablation_list):
            if true_ptm not in p['masks']:
                continue
            mask = ablation_masks[prompted_ptm][i]
            yt_full = p['masks'][true_ptm]
            yts.append(yt_full[mask])
            yss.append(ablation_predictions[prompted_ptm][i])
        if not yts:
            cross_auc.loc[true_ptm, prompted_ptm] = float('nan')
            continue
        yt = np.concatenate(yts).astype(np.int8)
        ys = np.concatenate(yss).astype(np.float32)
        if yt.sum() == 0:
            cross_auc.loc[true_ptm, prompted_ptm] = float('nan')
            continue
        try:
            cross_auc.loc[true_ptm, prompted_ptm] = float(roc_auc_score(yt, ys))
        except ValueError:
            cross_auc.loc[true_ptm, prompted_ptm] = float('nan')

print('\nCross-instruction AUC matrix (rows = true PTM type, cols = prompted PTM type):')
print(cross_auc.round(4).to_string())

In [ ]:
# Heatmap of the cross-task AUC matrix and a diagonal-vs-off-diagonal summary.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(
    cross_auc.astype(float), annot=True, fmt='.3f', cmap='YlGnBu',
    vmin=0.5, vmax=1.0, ax=axes[0], cbar_kws={'label': 'AUC'},
)
axes[0].set_xlabel('Prompted PTM type')
axes[0].set_ylabel('True PTM type')
axes[0].set_title('Cross-instruction AUC matrix')

diag = np.array([cross_auc.loc[ptm, ptm] for ptm in PTM_INSTRUCTIONS])
off_diag_means = []
labels = []
for ptm in PTM_INSTRUCTIONS:
    others = [cross_auc.loc[ptm, p] for p in PTM_INSTRUCTIONS if p != ptm]
    others = [v for v in others if not (isinstance(v, float) and math.isnan(v))]
    off_diag_means.append(np.mean(others) if others else float('nan'))
    labels.append(ptm)

x = np.arange(len(labels))
axes[1].bar(x - 0.2, diag, width=0.4, label='Matched instruction (diagonal)', color='#3a7d44')
axes[1].bar(x + 0.2, off_diag_means, width=0.4, label='Mismatched instruction (off-diagonal mean)', color='#c44536')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel('AUC')
axes[1].set_title('Matched vs. mismatched instruction (per true PTM type)')
axes[1].set_ylim(0.4, 1.0)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

fig.tight_layout()
ablation_plot_path = os.path.join(EVAL_DIR, 'cross_instruction_ablation.png')
fig.savefig(ablation_plot_path, dpi=150)
plt.show()
print('Saved:', ablation_plot_path)

In [ ]:
# Instruction-following rate: fraction of (protein, window) pairs whose
# generated raw output differs across PTM-type prompts. Recompute on the
# raw window completions rather than the consensus aggregation.
# To keep this cheap we use a smaller sub-sample of proteins.
follow_sample = ablation_list[:min(50, len(ablation_list))]
print(f'Computing instruction-following rate on {len(follow_sample)} proteins...')

window_outputs: Dict[Tuple[int, int], Dict[str, str]] = {}
for prompted_ptm in PTM_INSTRUCTIONS:
    jobs = []
    for prot_idx, p in enumerate(follow_sample):
        for start, w_seq in make_windows(p['sequence']):
            jobs.append((prot_idx, start, w_seq))
    for i in tqdm(range(0, len(jobs), BATCH_SIZE), desc=f'follow/{prompted_ptm}'):
        batch = jobs[i:i + BATCH_SIZE]
        prompts = [build_prompt(w_seq, prompted_ptm) for _, _, w_seq in batch]
        outs = generate_batch(prompts)
        for (prot_idx, start, w_seq), comp in zip(batch, outs):
            window_outputs.setdefault((prot_idx, start), {})[prompted_ptm] = comp.strip()

n_windows = len(window_outputs)
n_all_same = 0
n_at_least_one_diff = 0
for _, outs in window_outputs.items():
    unique = set(outs.values())
    if len(unique) == 1:
        n_all_same += 1
    else:
        n_at_least_one_diff += 1

instruction_follow_rate = n_at_least_one_diff / n_windows if n_windows else float('nan')
print()
print(f'Instruction-following rate (fraction of windows whose output differs across prompts):')
print(f'  {n_at_least_one_diff:,} / {n_windows:,}  =  {instruction_follow_rate * 100:.2f}%')
print(f'  windows with identical output across all three prompts: {n_all_same:,}')

## 10. Persist Artifacts

Four artifacts are written to disk and uploaded to the Hugging Face repository:

1. **`inference_config.json`** — the inference-time configuration: window size, stride, per-PTM-type calibrated threshold, the PTM-type instruction templates, and maximum new tokens. Downstream inference code loads this file directly.
2. **`metrics_summary.json`** — the complete numerical results from this run, including the calibration AUCs, per-PTM-type test metrics, and the cross-instruction ablation results.
3. **`per_residue_breakdown.csv`** — the per-residue-type breakdown table.
4. **`cross_instruction_auc.csv`** — the full cross-instruction AUC matrix.

In [ ]:
inference_config = {
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'prompt_template': PROMPT_TEMPLATE,
    'instructions': PTM_INSTRUCTIONS,
    'consensus_thresholds': {ptm: float(t) for ptm, t in LOCKED_THRESHOLDS.items()},
    'output_format': 'Sites=<X1,X2,...> where Xi is one-letter residue + 1-indexed position within the window',
}
inf_cfg_path = os.path.join(EVAL_DIR, 'inference_config.json')
with open(inf_cfg_path, 'w') as f:
    json.dump(inference_config, f, indent=2)
print('Saved:', inf_cfg_path)
print(json.dumps(inference_config, indent=2))

In [ ]:
summary = {
    'model_repo': HF_REPO_ID,
    'data_csv': PTM_DATA_CSV,
    'split_seed': SPLIT_SEED,
    'split_ratios': {'train': TRAIN_RATIO, 'calibration': CAL_RATIO, 'test': TEST_RATIO},
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'calibration': {},
    'test': {},
    'cross_instruction_auc': cross_auc.astype(float).to_dict(),
    'instruction_follow_rate': float(instruction_follow_rate) if not math.isnan(instruction_follow_rate) else None,
}

for ptm, r in calibration_results.items():
    summary['calibration'][ptm] = {
        'n_proteins': int(len(r['records'])),
        'n_residues_evaluated': r['n_residues'],
        'positive_prevalence': r['prevalence'],
        'roc_auc': r['auc'],
        'locked_threshold': r['threshold'],
        'f1_at_threshold': r['f1'],
        'precision_at_threshold': r['precision'],
        'recall_at_threshold': r['recall'],
    }

for ptm, r in test_results.items():
    summary['test'][ptm] = {
        'n_proteins': int(len(r['records'])),
        'n_residues_evaluated': r['n_residues'],
        'positive_prevalence': r['prevalence'],
        'threshold_applied': r['threshold'],
        'accuracy': r['accuracy'],
        'precision': r['precision'],
        'recall': r['recall'],
        'specificity': r['specificity'],
        'f1': r['f1'],
        'roc_auc': r['auc'],
        'confusion_matrix': r['confusion_matrix'],
    }

summary['per_residue_breakdown'] = per_residue_df.to_dict(orient='records')

summary_path = os.path.join(EVAL_DIR, 'metrics_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', summary_path)

per_residue_df.to_csv(os.path.join(EVAL_DIR, 'per_residue_breakdown.csv'), index=False)
cross_auc.to_csv(os.path.join(EVAL_DIR, 'cross_instruction_auc.csv'))
print('Saved per_residue_breakdown.csv and cross_instruction_auc.csv')

## 11. Generate and Push the Model Card

A `README.md` is generated documenting the multi-PTM inference architecture, the calibration and test methodology, the per-PTM-type metrics, and the cross-instruction ablation. It includes a self-contained reference implementation that takes a PTM type as input.

In [ ]:
def pct(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return '—'
    return f'{x * 100:.2f}%'


def auc_str(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return '—'
    return f'{x:.3f}'


# Build the per-PTM-type results table.
result_rows = []
for ptm in PTM_INSTRUCTIONS:
    cal = calibration_results.get(ptm)
    tst = test_results.get(ptm)
    if cal is None or tst is None:
        continue
    cm = tst['confusion_matrix']
    result_rows.append(
        f"| {ptm} | {tst['n_residues']:,} | {pct(tst['prevalence'])} "
        f"| {auc_str(cal['auc'])} | {tst['threshold']:.3f} | {auc_str(tst['auc'])} "
        f"| {pct(tst['accuracy'])} | {pct(tst['precision'])} | {pct(tst['recall'])} "
        f"| {pct(tst['specificity'])} | {pct(tst['f1'])} "
        f"| {cm['TN']:,} / {cm['FP']:,} / {cm['FN']:,} / {cm['TP']:,} |"
    )

results_table = (
    '| PTM type | Test residues | Positive prevalence | Calibration AUC | Locked t | Test AUC | Accuracy | Precision | Recall | Specificity | F1 | TN / FP / FN / TP |\n'
    '|---|---|---|---|---|---|---|---|---|---|---|---|\n'
    + '\n'.join(result_rows)
)

# Per-residue-type breakdown table.
per_residue_rows_md = []
for row in per_residue_df.to_dict(orient='records'):
    per_residue_rows_md.append(
        f"| {row['ptm_type']} | {row['residue']} | {row['n_residues']:,} | {row['n_positive']:,} "
        f"| {auc_str(row['auc'])} | {pct(row['accuracy'])} | {pct(row['precision'])} "
        f"| {pct(row['recall'])} | {pct(row['specificity'])} | {pct(row['f1'])} |"
    )
per_residue_table_md = (
    '| PTM type | Residue | Total | Positive | AUC | Accuracy | Precision | Recall | Specificity | F1 |\n'
    '|---|---|---|---|---|---|---|---|---|---|\n'
    + '\n'.join(per_residue_rows_md)
)

# Cross-instruction AUC matrix as Markdown.
cross_cols = ' | '.join(list(PTM_INSTRUCTIONS))
cross_rows_md = []
for true_ptm in PTM_INSTRUCTIONS:
    cells = []
    for prompted_ptm in PTM_INSTRUCTIONS:
        v = cross_auc.loc[true_ptm, prompted_ptm]
        cells.append(auc_str(v))
    cross_rows_md.append(f"| **{true_ptm}** | " + ' | '.join(cells) + ' |')
cross_table_md = (
    f'|  | {cross_cols} |\n'
    f'|---|' + '|'.join(['---'] * len(PTM_INSTRUCTIONS)) + '|\n'
    + '\n'.join(cross_rows_md)
)

In [ ]:
card = f'''---
base_model: GreatCaptainNemo/ProLLaMA_Stage_1
tags:
  - protein
  - ptm
  - methylation
  - phosphorylation
  - ubiquitination
  - lora
  - peft
library_name: transformers
pipeline_tag: text-generation
---

# PTM-LLaMA

LoRA fine-tune of [`GreatCaptainNemo/ProLLaMA_Stage_1`](https://huggingface.co/GreatCaptainNemo/ProLLaMA_Stage_1) instruction-tuned to predict post-translational modification (PTM) sites for three PTM types in a single adapter: **methylation**, **phosphorylation**, and **ubiquitination**.

## Task

Given a 21-residue peptide window and a PTM-type instruction, the model generates the list of modified positions in the format `Sites=<R5,D12,...>` (residue letter + 1-indexed position within the window). The PTM type to predict is selected by the instruction prompt; the output format is shared across PTM types.

## Inference architecture

Full-protein inference for a single PTM type proceeds in three steps:

1. **Sliding windows.** A 21-residue window is slid across the input sequence with stride 5; a tail window is appended so the final residues are covered.
2. **Per-window generation.** The model generates `Sites=<...>` for each window, prompted with the PTM-type instruction. Each predicted residue letter is validated against the window sequence at the indicated position; mismatches are discarded. Validated predictions are mapped to full-protein coordinates and accumulated into a per-residue **consensus score**, defined as `(# windows predicting the residue as a site) / (# windows covering the residue)`.
3. **Thresholding.** A per-PTM-type F1-optimal threshold (derived from the held-out calibration set) is applied to the consensus scores.

The per-PTM-type thresholds, windowing parameters, and prompt template are persisted in [`inference_config.json`](./inference_config.json).

### Reference implementation

```python
import re, json, torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = "{HF_REPO_ID}"

cfg = json.load(open(hf_hub_download(REPO, "inference_config.json")))
tok = AutoTokenizer.from_pretrained(REPO)
if tok.pad_token is None:
    tok.pad_token = tok.unk_token
mdl = AutoModelForCausalLM.from_pretrained(
    REPO, torch_dtype=torch.float16, device_map="auto"
).eval()

SITE_RE  = re.compile(r"^([A-Z])(\\d+)$")
SITES_RE = re.compile(r"Sites=<([^>]*)>")


@torch.no_grad()
def predict_sites(seq: str, ptm_type: str):
    """Predict PTM-site positions in a full protein for a given PTM type.

    Returns sites like ['K161', 'S203'] in 1-indexed full-protein coords.
    """
    if ptm_type not in cfg["instructions"]:
        raise ValueError(f"Unknown PTM type {{ptm_type}}; supported: {{list(cfg['instructions'])}}")
    w, s = cfg["window_size"], cfg["stride"]
    t = cfg["consensus_thresholds"][ptm_type]
    instruction = cfg["instructions"][ptm_type]
    L = len(seq)

    if L <= w:
        starts = [0]
    else:
        starts = list(range(0, L - w + 1, s))
        if starts[-1] + w < L:
            starts.append(L - w)

    covered, predicted = [0] * L, [0] * L
    for st in starts:
        win = seq[st:st + w]
        prompt = cfg["prompt_template"].format(instruction=instruction, input=f"Seq=<{{win}}>")
        enc = tok(prompt, return_tensors="pt").to(mdl.device)
        out = mdl.generate(
            **enc, max_new_tokens=cfg["max_new_tokens"], do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
        text = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

        for i in range(st, min(st + w, L)):
            covered[i] += 1
        m = SITES_RE.search(text)
        if not m:
            continue
        for part in m.group(1).split(","):
            mm = SITE_RE.match(part.strip())
            if not mm:
                continue
            letter, pos_local = mm.group(1), int(mm.group(2))
            pos_full = st + pos_local
            if 1 <= pos_full <= L and seq[pos_full - 1] == letter:
                predicted[pos_full - 1] += 1

    return [f"{{seq[i]}}{{i + 1}}" for i in range(L)
            if covered[i] > 0 and predicted[i] / covered[i] >= t]


# Example usage.
sites = predict_sites(
    "MASDEGKLFVGGLSFDTNEQALEQVFSKYGQISEVVVVKDRETQRSRGFGFVTFENIDDAKDAMMAMNGK",
    ptm_type="Phosphorylation",
)
print(sites)
```

## Training

- **Base model:** `GreatCaptainNemo/ProLLaMA_Stage_1`
- **Method:** LoRA SFT via `trl.SFTTrainer` + `peft.LoraConfig`
- **LoRA config:** r=64, alpha=128, dropout=0.05, target modules = q,k,v,o,gate,down,up_proj
- **Optimizer:** AdamW, lr=3e-4, cosine schedule, warmup=40 steps, max_grad_norm=1.0
- **Batching:** per-device batch 16 × grad_accum 8 (effective 128) at bf16, max_seq_length 2048
- **Epochs:** up to 8, with `EarlyStoppingCallback(patience=3)` on `eval_loss` and `load_best_model_at_end=True`
- **Source data:** `datasets/all_ptm_sites_site_level.csv` (long format: one row per annotated PTM site across methylation, phosphorylation, and ubiquitination)
- **Split:** protein-level partition with seed `{SPLIT_SEED}` and ratios {TRAIN_RATIO:.2f} / {CAL_RATIO:.2f} / {TEST_RATIO:.2f} (train / calibration / test). All annotations of a given `uniprot_id` are assigned to a single split.
- **Training distribution:** natural — the relative training-set abundance across PTM types reflects the natural distribution of annotated sites in the source data (phosphorylation ≫ ubiquitination ≫ methylation). Each `(protein, PTM_type)` record contributes all sliding windows over its sequence, including windows with no in-window site of that PTM type as in-context negatives.

![training_loss](./training_loss.png)

## Evaluation methodology

Two disjoint protein-level partitions are used to separate per-PTM-type threshold selection from final metric reporting:

- **Calibration set** ({len(cal_proteins):,} proteins): the F1-optimal threshold over the per-residue consensus ROC is selected per PTM type.
- **Test set** ({len(test_proteins):,} proteins): the calibration-derived per-PTM-type thresholds are applied without modification; the metrics below are unbiased point estimates of generalization.

### Per-PTM-type results

{results_table}

![test_metrics](./test_metrics.png)

### Per-residue-type breakdown on test

{per_residue_table_md}

### Cross-instruction ablation

For a sub-sample of test proteins, sliding-window inference was run three times — once per PTM-type instruction — on the same protein sequences. AUC is reported against each PTM type's ground-truth labels.

A working instruction-tuned model should have higher AUC on the diagonal (matched instruction) than off-diagonal (mismatched instruction).

{cross_table_md}

Instruction-following rate (fraction of windows whose output differs across the three prompts on a 50-protein sub-sample): **{instruction_follow_rate * 100:.2f}%**.

![cross_instruction_ablation](./cross_instruction_ablation.png)

![calibration_threshold_sweep](./calibration_threshold_sweep.png)

## Limitations

- **Window-local outputs.** The model emits positions inside a 21-residue window. Full-protein predictions are produced by the sliding-window aggregation described in *Inference architecture*; very short proteins (`length < 21`) are scored as a single window.
- **No structural context.** The model sees only primary sequence; structurally-disfavored false positives cannot be filtered without external 3D information.
- **Three PTM types only.** Predictions are well-defined for methylation, phosphorylation, and ubiquitination. Generalization to other PTM types is not evaluated.

## Reproduction

Execute `training/train_ptm_llama.ipynb` followed by `evaluation/evaluate_ptm_llama.ipynb` from the source repository. Both notebooks are self-contained and intended for execution in Google Colab. The protein-level split is deterministic given `SPLIT_SEED = {SPLIT_SEED}`.
'''

card_path = os.path.join(EVAL_DIR, 'README.md')
with open(card_path, 'w') as f:
    f.write(card)
print('Wrote', card_path)
print('First 2000 chars:\n')
print(card[:2000])

In [ ]:
if PUSH_MODEL_CARD:
    eval_artifacts = [
        'README.md',
        'inference_config.json',
        'metrics_summary.json',
        'per_residue_breakdown.csv',
        'cross_instruction_auc.csv',
        'calibration_threshold_sweep.png',
        'test_metrics.png',
        'cross_instruction_ablation.png',
    ]
    api = HfApi()
    api.upload_folder(
        folder_path=EVAL_DIR,
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
        allow_patterns=eval_artifacts,
        delete_patterns=eval_artifacts,
        commit_message='Push evaluation artifacts and model card',
    )
    print('Model card pushed to', f'https://huggingface.co/{HF_REPO_ID}')
else:
    print('PUSH_MODEL_CARD=False; skipped Hub upload.')

## Output Artifacts

The `eval_outputs/` directory contains:
- `inference_config.json` — inference-time configuration (window size, stride, per-PTM-type thresholds, prompt template, instructions)
- `metrics_summary.json` — full numerical results: per-PTM-type calibration AUC, test metrics, per-residue breakdown, cross-instruction AUC matrix, instruction-following rate
- `per_residue_breakdown.csv` — per-residue-type metrics on test
- `cross_instruction_auc.csv` — the full cross-instruction AUC matrix
- `calibration_threshold_sweep.png`, `test_metrics.png`, `cross_instruction_ablation.png` — figures
- `README.md` — model card uploaded to the Hugging Face repository